## Convert to upload checkpoints

In [1]:
from pathlib import Path
import zipfile

In [2]:
# Thư mục chứa tất cả checkpoint
SOURCE_DIR = Path(
    r"D:\Event-Extraction\model\Qwen3-4B\checkpoint-VHE"
)

# Thư mục và file ZIP đầu ra
OUTPUT_DIR = SOURCE_DIR / "validation_adapters"
ZIP_PATH = SOURCE_DIR / "validation_adapters.zip"

# File cần thiết để load LoRA/QLoRA adapter
REQUIRED_FILES = [
    "adapter_config.json",
    "adapter_model.safetensors",
]

In [3]:
# Tự tìm checkpoint và sắp xếp theo step
checkpoints = sorted(
    [
        path for path in SOURCE_DIR.iterdir()
        if path.is_dir()
        and path.name.startswith("checkpoint-")
        and path.name.removeprefix("checkpoint-").isdigit()
    ],
    key=lambda path: int(path.name.removeprefix("checkpoint-")),
)

if not checkpoints:
    raise FileNotFoundError(
        f"Không tìm thấy checkpoint trong {SOURCE_DIR}"
    )

# Xóa ZIP cũ để tránh giữ dữ liệu lỗi thời
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

included = []
skipped = []


In [4]:
with zipfile.ZipFile(
    ZIP_PATH, mode="w", compression=zipfile.ZIP_STORED
) as archive:

    for checkpoint in checkpoints:
        files = {
            name: checkpoint / name
            for name in REQUIRED_FILES
        }

        missing = [
            name for name, path in files.items()
            if not path.is_file()
        ]

        if missing:
            skipped.append((checkpoint.name, missing))
            print(f"[SKIP] {checkpoint.name}: thiếu {missing}")
            continue

        for name, path in files.items():
            # Giữ nguyên thư mục checkpoint bên trong ZIP
            archive.write(
                path,
                arcname=f"{checkpoint.name}/{name}"
            )

        included.append(checkpoint.name)
        print(f"[OK] {checkpoint.name}")

print("\n========== SUMMARY ==========")
print(f"Tổng checkpoint tìm thấy : {len(checkpoints)}")
print(f"Checkpoint được đóng gói  : {len(included)}")
print(f"Checkpoint bị bỏ qua      : {len(skipped)}")
print(f"File ZIP                  : {ZIP_PATH}")
print(f"Dung lượng ZIP             : {ZIP_PATH.stat().st_size / 1024**3:.3f} GB")

if skipped:
    print("\nCheckpoint bị bỏ qua:")
    for name, missing in skipped:
        print(f"  {name}: thiếu {', '.join(missing)}")

[OK] checkpoint-409
[OK] checkpoint-818
[OK] checkpoint-1227
[OK] checkpoint-1636
[OK] checkpoint-2045
[OK] checkpoint-2454
[OK] checkpoint-2863
[OK] checkpoint-3272
[OK] checkpoint-3681
[OK] checkpoint-4090

========== SUMMARY ==========
Tổng checkpoint tìm thấy : 10
Checkpoint được đóng gói  : 10
Checkpoint bị bỏ qua      : 0
File ZIP                  : D:\Event-Extraction\model\Qwen3-4B\checkpoint-VHE\validation_adapters.zip
Dung lượng ZIP             : 1.231 GB
